<a href="https://colab.research.google.com/github/Saurabh-Kurhade-Patil/GenAI/blob/main/langchain_VectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community requests==2.32.4 opentelemetry-api<=1.42.1 langchain_huggingface

/bin/bash: line 1: =1.42.1: No such file or directory


In [9]:
%pip install -qU langchain-community langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu beautifulsoup4 langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 6.8 MB/s eta 0:00:00


In [46]:
import os
os.environ["HF_TOKEN"] = 'hf_'

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vector = embeddings.embed_query(
    "What is the price of the MacBook?"
)

print("Embedding dimensions:", len(vector))
print("First 10 values:", vector[:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions: 384
First 10 values: [-0.008308391086757183, 0.06082596629858017, 0.016990363597869873, 0.022889234125614166, 0.012871423736214638, 0.015779243782162666, -0.015377291478216648, 0.05772567167878151, 0.10423984378576279, 0.06994591653347015]


In [13]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

In [15]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [16]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [20]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=docs,
    embedding = embeddings
)

In [29]:
vector_store.index.ntotal

5

In [37]:
query = "who is death overs best bowler?"

results = vector_store.similarity_search(
    query,
    k=1
)
print(results[0].page_content)

Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.


In [45]:
vector_store.similarity_search_with_score(
    query='Who among these are a best bowler?',
    k=2,
    filter = {"team": "Mumbai Indians"}
)

[(Document(id='acc208c6-b5c1-43bd-b717-a61ad4b81e72', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  np.float32(0.8159027)),
 (Document(id='b750f10c-0ba7-46d7-b4f4-953a05dd65b7', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  np.float32(0.93092155))]